In [ ]:
# default_exp cleaning

# Data Preparation
> This notebook loads the original CSV file with the reflectances and it has the functions to clean data:
> * Remove negative
> * Remove bad data <br>
>
> It also saves a final version of the data frame as PICKLE format


In [ ]:
#hide
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib
import matplotlib.pyplot as plt

import plotly
import plotly.express as px
import plotly.graph_objects as go

from sklearn import decomposition, datasets, cluster
from sklearn.model_selection import train_test_split

import pyarrow

import pandas as pd

from WaterClassification.core import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Loading original CSV
The input data shall be a CSV file with the following header information
* Id - Unique identifier for the measurement (that will be used throughout the notebooks)
* Projeto
* Rio/ Bacia
* Ponto
* SPM
* Reflectances from 320 to 950 in 1nm steps

In [ ]:
df_raw = pd.read_csv('data/reflectances_3.csv')
df_raw.head(n=2)

,Id,Projeto,Rio/ Bacia,Ponto,DATA,SPM,320,321,322,323,...,941,942,943,944,945,946,947,948,949,950
0,0,AQUASENSE,Paranoa,DO-P_02,5/8/2015,12.4,0.000735377,0.000655,0.000720,0.000764,...,0.001205,0.001175,0.001149,0.001128,0.001130,0.001140,0.001146,0.001146,0.001137,0.001118
1,1,AQUASENSE,Paranoa,DO-P_03,5/8/2015,6.2,0.000474554,0.000429,0.000513,0.000573,...,0.000726,0.000700,0.000677,0.000656,0.000684,0.000731,0.000773,0.000774,0.000721,0.000665


## Data Cleaning
Some of the original measurements will be discarded due to inconsistencies or bad values. <br>
Those inconsistencies and respective Ids will be detailed bellow:

In [ ]:
#export
def delete_rows(df, ids):
    "This function cleans the informed ids from the dataframe"
    return df[~df.Id.isin(ids)]

def show_rows(df, ids, columns=None):
    "This function shows only the rows with the informed ids"
    columns = df.columns if columns is None else columns
    return df[df.Id.isin(ids)][columns]

### Bad Measurements ("Não rodou" and "ruim")
There are 6 measurements that have been marked with with these flags in the CSV

In [ ]:
def get_bad_measurements(df, bad_flags=['não rodou', 'ruim'], flag_col='320'):
    "Cleans the rows in dataframe df that contains the bad_flags"
    
    return df[df[flag_col].isin(bad_flags)]['Id'].to_list()

In [ ]:
show_rows(df_raw, get_bad_measurements(df_raw))

,Id,Projeto,Rio/ Bacia,Ponto,DATA,SPM,320,321,322,323,...,941,942,943,944,945,946,947,948,949,950
925,925,NaN,Maroni,GA4,março,28.0,ruim,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
931,931,NaN,Maroni,SH hyb 1,março,28.0,não rodou,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
932,932,NaN,Maroni,SH hyb 2,março,31.3,não rodou,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
933,933,NaN,Maroni,SH Hyb 3,março,38.0,não rodou,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
934,934,NaN,Maroni,SH hyb 4,março,41.3,não rodou,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#hide
df = delete_rows(df_raw, get_bad_measurements(df_raw))

### Missing reflectance values
The following measurements have missing values between 375 and 940 nm.

In [ ]:
# export
def get_missing_values(df, columns):
    "Get rows with missing values in a specified range of wavelengths"
    
    # get Ids that have missing values between two wavelenghts
    sub_df = df.set_index('Id')[columns]
    sub_df.dropna(axis=0, inplace=True)
    
    return df[~df.Id.isin(sub_df.index)]['Id'].to_list()

In [ ]:
get_missing_values(df, wavelength_range(375, 940))

[213, 358, 370, 383, 391, 415, 541]

In [ ]:
#hide
df = delete_rows(df, get_missing_values(df, wavelength_range(375, 940)))

### Missing SPM values
The following IDs have no SPM values.

In [ ]:
get_missing_values(df, ['SPM'])

[487, 498, 499, 542, 543, 544, 547, 551]

In [ ]:
#hide
df = delete_rows(df, get_missing_values(df, ['SPM']))

### Negative Reflectances

In [ ]:
#export
def get_negative_values(df, columns):
    "Get rows with negative values in the specified columns. NaN will be considered negative." 
    sub_df = df.set_index('Id')[columns]
    negatives = (sub_df<0).any(axis=1)
    
    return df[negatives]['Id'].to_list()

In [ ]:
len(get_negative_values(df, wavelength_range(375, 940)))

51

In [ ]:
df = delete_rows(df, get_negative_values(df, wavelength_range(375, 940)))

## Final Format
To put the dataframe in a standard format for processing, we will drop all unwanted columns. <br>
The following function mantains only the desired columns within a dataframe.

In [ ]:
#export
def keep_columns(df, wanted_columns):
    "Keeps the wanted_columns and drop the rest." 
    drop_columns = df.columns[~df.columns.isin(wanted_columns)]
    return df.drop(columns=drop_columns)

In [ ]:
df = keep_columns(df, wanted_columns = ['Id', 'Projeto', 'Rio/ Bacia', 'Ponto', 'DATA', 'SPM'] + wavelength_range(375, 940))

In [ ]:
df

,Id,Projeto,Rio/ Bacia,Ponto,DATA,SPM,375,376,377,378,...,931,932,933,934,935,936,937,938,939,940
0,0,AQUASENSE,Paranoa,DO-P_02,5/8/2015,12.4,0.002848,0.002881,0.002940,0.002960,...,0.001061,0.001130,0.001191,0.001205,0.001232,0.001260,0.001204,0.001170,0.001190,0.001199
1,1,AQUASENSE,Paranoa,DO-P_03,5/8/2015,6.2,0.002350,0.002381,0.002434,0.002451,...,0.000623,0.000651,0.000666,0.000652,0.000671,0.000715,0.000715,0.000718,0.000726,0.000727
2,2,AQUASENSE,Paranoa,DO-P_04,5/8/2015,4.3,0.001604,0.001642,0.001698,0.001709,...,0.000401,0.000446,0.000495,0.000530,0.000564,0.000588,0.000571,0.000555,0.000543,0.000525
3,3,AQUASENSE,Paranoa,DO-P_05,5/8/2015,3.8,0.002134,0.002160,0.002205,0.002218,...,0.000432,0.000447,0.000454,0.000439,0.000449,0.000478,0.000478,0.000477,0.000474,0.000467
4,4,AQUASENSE,Paranoa,DO-P_08,5/8/2015,5.5,0.002626,0.002675,0.002752,0.002769,...,0.001148,0.001216,0.001257,0.001239,0.001268,0.001295,0.001233,0.001201,0.001233,0.001255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
998,998,NaN,Sao Francisco,P11_Rio SF_Barra_2S,NaN,197.7,0.089195,0.091395,0.093284,0.095102,...,3.426929,3.351447,3.423282,3.509999,3.435005,3.407089,3.419072,3.391828,3.328344,3.294187
999,999,NaN,Sao Francisco,P11_Rio SF_Barra_3S,NaN,239.7,0.007271,0.007361,0.007413,0.007456,...,0.019283,0.019564,0.021137,0.021147,0.020443,0.020663,0.020504,0.019593,0.018956,0.018590
1000,1000,NaN,Sao Francisco,12_Rio SF_Morpará_1S,NaN,210.3,0.007478,0.007578,0.007643,0.007698,...,0.017353,0.017290,0.017896,0.017595,0.016925,0.016810,0.016511,0.015838,0.015305,0.014923
1001,1001,NaN,Sao Francisco,12_Rio SF_Morpará_2S,NaN,171.0,0.008073,0.008169,0.008228,0.008278,...,0.016026,0.015972,0.016600,0.016376,0.015806,0.015711,0.015425,0.014781,0.014261,0.013880


## Load and Clean data

In [ ]:
def load_and_clean(df, flag_col='320', 
                   bad_flags=['não rodou', 'ruim'],
                   wl_range = wavelength_range(375, 940),
                   clean_missing=True,
                   clean_neg=True,
                   clean_SPM_column='SPM',
                   display_ids=10,
                   base_columns=['Id', 'Projeto', 'Rio/ Bacia', 'Data', 'SPM']
                  ):
    "Load the CSV and apply the cleaning process accordingly."
    # if df is a string, than load the CSV from disk, otherwise, assume it is the desired dataframe
    if isinstance(df, str):
        df = pd.read_csv(df)
    
    # cleaning bad measurements
    if flag_col is not None and bad_flags is not None:
        ids = get_bad_measurements(df, bad_flags=bad_flags, flag_col=flag_col)
        print(f'Deleting {len(ids)} rows with bad_flags: {ids[:display_ids]}')
        df = delete_rows(df, ids)
    
    # cleaning missing values
    if clean_missing:
        ids = get_missing_values(df, wl_range)
        print(f'Deleting {len(ids)} rows with missing values (from {wl_range[0]} to {wl_range[-1]}nm): {ids[:display_ids]}')
        df = delete_rows(df, ids)
        
    if clean_neg:
        # clean negative values in the wavelength range
        ids = get_negative_values(df, wl_range)
        print(f'Deleting {len(ids)} rows with negative values (from {wl_range[0]} to {wl_range[-1]}nm): {ids[:display_ids]}')
        df = delete_rows(df, ids)
            
    if clean_SPM_column:
        try:
            ids = get_missing_values(df, clean_SPM_column)
            print(f'Deleting {len(ids)} rows with missing SPM values in {clean_SPM_column}: {ids[:display_ids]}')
            df = delete_rows(df, ids)
        
        except:
            print(f'Could not find SPM values in {clean_SPM_column}')
         
    # Drop unwanted columns
    wanted_columns = base_columns + wl_range

    return keep_columns(df, wanted_columns)

In [ ]:
load_and_clean('data/reflectances_3.csv')

Deleting 5 rows with bad_flags: [925, 931, 932, 933, 934]
Deleting 7 rows with missing values (from 375 to 940nm): [213, 358, 370, 383, 391, 415, 541]
Deleting 51 rows with negative values (from 375 to 940nm): [590, 600, 619, 653, 667, 724, 725, 726, 727, 729]
Deleting 8 rows with missing SPM values in SPM: [487, 498, 499, 542, 543, 544, 547, 551]


,Id,Projeto,Rio/ Bacia,SPM,375,376,377,378,379,380,...,931,932,933,934,935,936,937,938,939,940
0,0,AQUASENSE,Paranoa,12.4,0.002848,0.002881,0.002940,0.002960,0.002974,0.003029,...,0.001061,0.001130,0.001191,0.001205,0.001232,0.001260,0.001204,0.001170,0.001190,0.001199
1,1,AQUASENSE,Paranoa,6.2,0.002350,0.002381,0.002434,0.002451,0.002463,0.002514,...,0.000623,0.000651,0.000666,0.000652,0.000671,0.000715,0.000715,0.000718,0.000726,0.000727
2,2,AQUASENSE,Paranoa,4.3,0.001604,0.001642,0.001698,0.001709,0.001718,0.001774,...,0.000401,0.000446,0.000495,0.000530,0.000564,0.000588,0.000571,0.000555,0.000543,0.000525
3,3,AQUASENSE,Paranoa,3.8,0.002134,0.002160,0.002205,0.002218,0.002226,0.002268,...,0.000432,0.000447,0.000454,0.000439,0.000449,0.000478,0.000478,0.000477,0.000474,0.000467
4,4,AQUASENSE,Paranoa,5.5,0.002626,0.002675,0.002752,0.002769,0.002781,0.002858,...,0.001148,0.001216,0.001257,0.001239,0.001268,0.001295,0.001233,0.001201,0.001233,0.001255
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
998,998,NaN,Sao Francisco,197.7,0.089195,0.091395,0.093284,0.095102,0.096961,0.099661,...,3.426929,3.351447,3.423282,3.509999,3.435005,3.407089,3.419072,3.391828,3.328344,3.294187
999,999,NaN,Sao Francisco,239.7,0.007271,0.007361,0.007413,0.007456,0.007547,0.007701,...,0.019283,0.019564,0.021137,0.021147,0.020443,0.020663,0.020504,0.019593,0.018956,0.018590
1000,1000,NaN,Sao Francisco,210.3,0.007478,0.007578,0.007643,0.007698,0.007805,0.007975,...,0.017353,0.017290,0.017896,0.017595,0.016925,0.016810,0.016511,0.015838,0.015305,0.014923
1001,1001,NaN,Sao Francisco,171.0,0.008073,0.008169,0.008228,0.008278,0.008382,0.008553,...,0.016026,0.015972,0.016600,0.016376,0.015806,0.015711,0.015425,0.014781,0.014261,0.013880


### Discarding curves with identified problems IDs: 329, 330 371, 383, 992, 993

In [ ]:
df = df.set_index('Id').drop(index=[329, 330, 371, 383, 992, 993, 999, 998, 346]).reset_index()

### Discarding curves with negative reflectances
We found a total of 53 measurements that contains negative reflectances
Most of them (40) are related to project THIAGO_CAMP3 in 2013. For now, the working dataframe will discard those measurements, as they interfere with the final clustering results. In the future we could try to fix these values or just shift them.

In [ ]:
df.set_index('Id', inplace=True)
negatives = ~((df.iloc[:, 6:]>=0).all(1))
# df[negatives]

In [ ]:
df = df.drop(index=df[negatives].index).reset_index()

### Saving final dataframe to feather format

In [ ]:
df.to_pickle('data/reflectances_3.pickle')

In [ ]:
# df = pd.read_pickle('data/reflectances_3.pickle')